In [ ]:
!pip install -q transformers accelerate sentencepiece torch tqdm

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model Loaded Successfully")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model Loaded Successfully


In [ ]:
import re

def generate_sql(question, schema):

    prompt = f"""
    ### Instruction:
    You are a text-to-SQL generator.

    Generate a SQL query using the schema.

    Return ONLY SQL.

    ### Schema:
    {schema}

    ### Question:
    {question}

    ### Response:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "### Response:" in response:
        response = response.split("### Response:")[-1]

    # Step 1: Try to extract content from the last markdown code block (```...```)
    # This regex looks for ``` followed by optional language (like sql) and captures content
    code_blocks = re.findall(r"```(?:[a-zA-Z0-9_\\-]+)?\\s*(.*?)\\s*```", response, re.DOTALL)

    if code_blocks:
        # Prioritize the last code block as it's often the main answer
        for block_content in reversed(code_blocks):
            stripped_block = block_content.strip()
            # Check if the stripped block content starts with a common SQL keyword
            if stripped_block.upper().startswith(('SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'ALTER', 'DROP', 'WITH')):
                return stripped_block
        # If no SQL-like block found in code_blocks, try to return the last one anyway if it's not empty
        last_block = code_blocks[-1].strip()
        if last_block:
            return last_block

    # Step 2: Fallback to finding a line that starts with SQL keywords outside a code block
    lines = response.split('\n')
    for line in lines:
        stripped_line = line.strip()
        if stripped_line.upper().startswith(('SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'ALTER', 'DROP', 'WITH')):
            return stripped_line

    # Step 3: If all else fails, return the original stripped response (should ideally not happen if model follows instructions)
    return response.strip()

In [ ]:
schema = """
CREATE TABLE singer (
    singer_id INTEGER,
    name TEXT,
    age INTEGER,
    PRIMARY KEY(singer_id)
);

CREATE TABLE concert (
    concert_id INTEGER,
    singer_id INTEGER,
    location TEXT,
    FOREIGN KEY(singer_id)
        REFERENCES singer(singer_id)
);
"""

In [ ]:
question = "How many singers are there?"

sql = generate_sql(
    question,
    schema
)

print(sql)

SELECT COUNT(*) FROM singer;


In [ ]:
import sqlite3

conn = sqlite3.connect("test.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE singer(
    singer_id INTEGER,
    name TEXT,
    age INTEGER
)
""")

cursor.execute("""
INSERT INTO singer VALUES
(1,'John',25),
(2,'Alice',30),
(3,'David',40)
""")

conn.commit()

print("Database Created")

Database Created


In [ ]:
def execute_sql(sql):

    try:
        conn = sqlite3.connect("test.db")

        cursor = conn.cursor()

        cursor.execute(sql)

        result = cursor.fetchall()

        conn.close()

        return result

    except Exception as e:
        return str(e)

In [ ]:
question = "How many singers are there?"

generated_sql = generate_sql(
    question,
    schema
)

print("Generated SQL:")
print(generated_sql)

print("\nResult:")

result = execute_sql(
    generated_sql
)

print(result)

Generated SQL:
SELECT COUNT(*) FROM singer;

Result:
[(3,)]


# Flow
 text-to-SQL workflow: 'Question -> Qwen2.5-Coder -> Generated SQL -> SQLite Execution -> Answer'

### 1. Question
This initial stage represents the user's input in natural language. It is the problem statement or the query that the user wants to resolve by interacting with a database. For instance, a user might ask, "How many singers are there?" This natural language question serves as the starting point for the entire text-to-SQL process. It's crucial because it defines the objective that the subsequent steps aim to fulfill.

### 2. Qwen2.5-Coder
This is the core of the text-to-SQL transformation. The Qwen2.5-Coder is an advanced large language model (LLM) specifically fine-tuned for code generation, including SQL. Upon receiving the natural language "Question" from the user, the Qwen2.5-Coder processes this input along with a provided database schema (which gives it context about the tables, columns, and relationships in the database). Its primary task is to interpret the intent behind the natural language query and translate it into a semantically equivalent SQL query. This step involves complex natural language understanding and code synthesis, leveraging the model's training to bridge the gap between human language and database query language.

### 3. Generated SQL
Following the processing by the Qwen2.5-Coder, the output is a syntactically correct and semantically appropriate SQL query. This SQL query is directly derived from the user's natural language question and the provided database schema. For instance, if the question was "How many singers are there?", the generated SQL might be `SELECT COUNT(*) FROM singer;`. This generated SQL is a crucial intermediate representation, as it is the executable instruction set that a database system understands. Its accuracy and efficiency are paramount for the overall success of the text-to-SQL system.

### 4. SQLite Execution
This stage involves taking the "Generated SQL" query and executing it against a SQLite database. The system connects to the database (in this case, 'test.db'), prepares the SQL query, and runs it. This is where the actual data retrieval or manipulation happens. The database engine processes the query, accesses the relevant tables, applies any filtering, aggregation, or joining operations, and produces a result set. This step is critical as it validates the correctness of the generated SQL and fetches the desired information from the database.

### 5. Answer
This is the ultimate output of the entire text-to-SQL workflow. After the `Generated SQL` is executed by `SQLite Execution`, the database returns a result set. This result set, often in a raw format (e.g., a list of tuples like `[(3,)]`), is the `Answer` to the original `Question`. For the example "How many singers are there?", the answer `[(3,)]` indicates that there are 3 singers. The system can then present this answer to the user in a readable format. This final step completes the cycle, providing the user with the requested information extracted from the database based on their natural language query.

```markdown
## Text-to-SQL Workflow

Workflow of a text-to-SQL system, transforming a natural language question into an actionable SQL query and ultimately providing an answer. The process is broken down into five distinct stages: Question, Qwen2.5-Coder, Generated SQL, SQLite Execution, and Answer. Each step is designed to demonstrate a clear progression and transformation of information.

### 1. Question
**Description:** This initial stage captures the user's intent in natural language. It is the raw problem statement or query that the user wishes to resolve against a database. For example, a user might pose a question like, "How many singers are there?"
**Transformation/Processing:** The natural language input is the starting point, defining the objective for the subsequent stages. No transformation occurs here, but it establishes the input for the next phase.

### 2. Qwen2.5-Coder
**Description:** This is the intelligent core of the system, leveraging an advanced large language model (LLM), Qwen2.5-Coder, specifically fine-tuned for code generation, including SQL. It receives the natural language question and contextual database schema.
**Transformation/Processing:** The Qwen2.5-Coder interprets the linguistic nuances and semantic intent of the natural language question. It then synthesizes this understanding with the provided database schema to generate a logically and syntactically correct SQL query. This is the crucial translation step from human language to database query language.

### 3. Generated SQL
**Description:** The direct output from the Qwen2.5-Coder is a complete SQL query. This query is the machine-readable instruction set corresponding to the user's original question. For our example, "How many singers are there?", the generated SQL would be `SELECT COUNT(*) FROM singer;`.
**Transformation/Processing:** This stage represents the conversion of natural language intent into a structured, executable database command. The output is a string of SQL code, ready for execution. No further transformation occurs here, but it is the critical intermediate representation.

### 4. SQLite Execution
**Description:** At this stage, the generated SQL query is executed against a target SQLite database (e.g., 'test.db'). This involves connecting to the database, preparing the query, and running it to retrieve or manipulate data.
**Transformation/Processing:** The database engine processes the SQL query, accessing relevant tables, applying filters, aggregations, or joins as specified. The database executes the command and produces a raw result set. This is where the database actively works on the data.

### 5. Answer
**Description:** This is the final output presented back to the user. It is the result set obtained from the SQLite Execution, directly addressing the original natural language question. For the query "How many singers are there?", the answer might be `[(3,)]`, indicating three singers.
**Transformation/Processing:** The raw result set from the database is processed (e.g., formatted for readability) and presented as the conclusive answer. This completes the cycle, delivering the requested information in a digestible format.

### Conclusion
This end-to-end workflow demonstrates a seamless and automated process for converting natural language questions into database queries and obtaining precise answers. By leveraging advanced LLM capabilities for SQL generation and integrating with standard database execution, the system effectively bridges the gap between human communication and structured data retrieval, providing a powerful and intuitive data access solution.
```

In [ ]:
# Spider Dataset
#     ↓
# Load Question
#     ↓
# Load Database Schema
#     ↓
# Generate SQL
#     ↓
# Execute SQL on SQLite DB
#     ↓
# Compare with Ground Truth

In [ ]:
!find spider_data -name "tables.json"
!find spider_data -name "dev.json"
!find spider_data -name "*.sqlite" | head

spider_data/spider_data/tables.json
spider_data/spider_data/dev.json
spider_data/__MACOSX/spider_data/database/theme_gallery/._theme_gallery.sqlite
spider_data/__MACOSX/spider_data/database/activity_1/._activity_1.sqlite
spider_data/__MACOSX/spider_data/database/student_1/._student_1.sqlite
spider_data/__MACOSX/spider_data/database/college_1/._college_1.sqlite
spider_data/__MACOSX/spider_data/database/behavior_monitoring/._behavior_monitoring.sqlite
spider_data/__MACOSX/spider_data/database/twitter_1/._twitter_1.sqlite
spider_data/__MACOSX/spider_data/database/browser_web/._browser_web.sqlite
spider_data/__MACOSX/spider_data/database/student_transcripts_tracking/._student_transcripts_tracking.sqlite
spider_data/__MACOSX/spider_data/database/city_record/._city_record.sqlite
spider_data/__MACOSX/spider_data/database/station_weather/._station_weather.sqlite


In [ ]:
import json

with open("spider_data/spider_data/dev.json", "r") as f:
    spider = json.load(f)

print("Examples:", len(spider))
print(spider[0])

Examples: 1034
{'db_id': 'concert_singer', 'query': 'SELECT count(*) FROM singer', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'singer'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'singer'], 'question': 'How many singers do we have?', 'question_toks': ['How', 'many', 'singers', 'do', 'we', 'have', '?'], 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []}, 'select': [False, [[3, [0, [0, 0, False], None]]]], 'where': [], 'groupBy': [], 'having': [], 'orderBy': [], 'limit': None, 'intersect': None, 'union': None, 'except': None}}


In [ ]:
print(spider[0])

{'db_id': 'concert_singer', 'query': 'SELECT count(*) FROM singer', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'singer'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'singer'], 'question': 'How many singers do we have?', 'question_toks': ['How', 'many', 'singers', 'do', 'we', 'have', '?'], 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []}, 'select': [False, [[3, [0, [0, 0, False], None]]]], 'where': [], 'groupBy': [], 'having': [], 'orderBy': [], 'limit': None, 'intersect': None, 'union': None, 'except': None}}


In [ ]:
with open("spider_data/spider_data/tables.json", "r") as f:
    tables = json.load(f)

print("Databases:", len(tables))

Databases: 166


In [ ]:
schema_lookup = {}

for db in tables:
    schema_lookup[db["db_id"]] = db

In [ ]:
def build_schema(db_id):

    db = schema_lookup[db_id]

    table_names = db["table_names_original"]

    column_names = db["column_names_original"]

    schema_text = ""

    for table_idx, table_name in enumerate(table_names):

        schema_text += f"\nTABLE {table_name}\n"

        for col_table_idx, col_name in column_names:

            if col_table_idx == table_idx:
                schema_text += f"  {col_name}\n"

    return schema_text

In [ ]:
db_id = spider[0]["db_id"]

schema = build_schema(db_id)

print(schema)


TABLE stadium
  Stadium_ID
  Location
  Name
  Capacity
  Highest
  Lowest
  Average

TABLE singer
  Singer_ID
  Name
  Country
  Song_Name
  Song_release_year
  Age
  Is_male

TABLE concert
  concert_ID
  concert_Name
  Theme
  Stadium_ID
  Year

TABLE singer_in_concert
  concert_ID
  Singer_ID



In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("Model Loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model Loaded


In [ ]:
import re
import torch

def generate_sql(question, schema):

    prompt = f"""You are an expert SQL generator.

    Database Schema:
    {schema}

    Question:
    {question}

    SQL:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Find first SELECT statement
    match = re.search(
        r"(SELECT[\s\S]*?)(?:\n\n|$)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        sql = match.group(1).strip()

        # Keep only first SQL statement
        if ";" in sql:
            sql = sql.split(";")[0] + ";"

        return sql

    return ""

In [ ]:
example = spider[0]

pred_sql = generate_sql(
    example["question"],
    build_schema(example["db_id"])
)

print(pred_sql)

SELECT COUNT(*) FROM singer


In [ ]:
example = spider[0]

question = example["question"]

schema = build_schema(
    example["db_id"]
)

pred_sql = generate_sql(
    question,
    schema
)

print("Question:")
print(question)

print("\nGenerated SQL:")
print(pred_sql)

print("\nGround Truth:")
print(example["query"])


Question:
How many singers do we have?

Generated SQL:
SELECT COUNT(*) FROM singer

Ground Truth:
SELECT count(*) FROM singer


In [ ]:
import sqlite3
import os
import subprocess # Needed for ls -R if db not found

# Define the database file path based on the current example's db_id
db_id = example["db_id"] # Should be 'concert_singer' from spider[0]
db_file_name = f"{db_id}.sqlite"
db_file_path = os.path.join("spider_data", "spider_data", "database", db_id, db_file_name)

print(f"Database path for execution: {db_file_path}")

if not os.path.exists(db_file_path):
    print(f"Error: Database file not found at {db_file_path}")
    print("Please ensure the Spider dataset databases are correctly extracted.")
    # Optional: list contents of the database directory for debugging
    base_db_dir = os.path.join("spider_data", "spider_data", "database")
    if os.path.exists(base_db_dir):
        print(f"Contents of {base_db_dir}:")
        ls_output = subprocess.run(["ls", "-R", base_db_dir], capture_output=True, text=True, check=False)
        print(ls_output.stdout)
    else:
        print(f"Base database directory {base_db_dir} not found.")
else:
    def execute_sql_spider(sql_query, db_file_path):
        try:
            conn = sqlite3.connect(db_file_path)
            cursor = conn.cursor()
            cursor.execute(sql_query)
            result = cursor.fetchall()
            conn.close()
            return result
        except Exception as e:
            return str(e)

    # Execute the generated SQL (which is ground truth in this case)
    execution_result = execute_sql_spider(pred_sql, db_file_path)

    print(f"\nExecution Result for Question: '{question}':")
    print(execution_result)
    print(f"Generated SQL: {pred_sql}")
    print(f"Ground Truth SQL: {example['query']}")


Database path for execution: spider_data/spider_data/database/concert_singer/concert_singer.sqlite

Execution Result for Question: 'How many singers do we have?':
[(6,)]
Generated SQL: SELECT COUNT(*) FROM singer
Ground Truth SQL: SELECT count(*) FROM singer


In [ ]:
import os

def get_db_path(db_id):

    return os.path.join(
        "spider_data",
        "spider_data",
        "database",
        db_id,
        f"{db_id}.sqlite"
    )

In [ ]:
import sqlite3

def run_sql(sql, db_path):

    try:

        conn = sqlite3.connect(db_path)

        cursor = conn.cursor()

        cursor.execute(sql)

        result = cursor.fetchall()

        conn.close()

        return result

    except Exception as e:

        return str(e)

In [ ]:
example = spider[0]

print(example)

{'db_id': 'concert_singer', 'query': 'SELECT count(*) FROM singer', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'singer'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'singer'], 'question': 'How many singers do we have?', 'question_toks': ['How', 'many', 'singers', 'do', 'we', 'have', '?'], 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []}, 'select': [False, [[3, [0, [0, 0, False], None]]]], 'where': [], 'groupBy': [], 'having': [], 'orderBy': [], 'limit': None, 'intersect': None, 'union': None, 'except': None}}


In [ ]:
example = spider[0]

db_path = get_db_path(
    example["db_id"]
)

gold_sql = example["query"]

pred_sql = generate_sql(
    example["question"],
    build_schema(example["db_id"])
)

gold_result = run_sql(
    gold_sql,
    db_path
)

pred_result = run_sql(
    pred_sql,
    db_path
)

print("Gold Result:")
print(gold_result)

print("\nPredicted Result:")
print(pred_result)

print("\nExecution Match:")
print(gold_result == pred_result)

Gold Result:
[(6,)]

Predicted Result:
[(6,)]

Execution Match:
True


In [ ]:
correct = 0

total = 50

for example in spider[:total]:

    try:

        db_id = example["db_id"]

        question = example["question"]

        gold_sql = example["query"]

        schema = build_schema(db_id)

        pred_sql = generate_sql(
            question,
            schema
        )

        db_path = get_db_path(db_id)

        gold_result = run_sql(
            gold_sql,
            db_path
        )

        pred_result = run_sql(
            pred_sql,
            db_path
        )

        if gold_result == pred_result:
            correct += 1

    except Exception:
        pass

print(
    f"Execution Accuracy: {correct/total:.4f}"
)

Execution Accuracy: 0.4000


In [ ]:
correct

20